In [43]:
import os
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Configuration ---
USE_IMPUTATION = True
WINSORIZE_QUANTILE = 0.99
LAG_DAYS = [1, 3, 7]
ROLL_WINDOWS = [3, 7]
TARGET_COL = "Number of Admissions"
WIND_DIR_COL = "wind_direction_10m_dominant (°)"
N_SPLITS = 5
N_TRIALS = 25 
RANDOM_STATE = 42

# --- 1. Feature Engineering with Wind Encoding ---
def engineer_features(df):
    df = df.copy()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.sort_values('Timestamp').reset_index(drop=True)
    
    # Time Features
    df["dow_sin"] = np.sin(2 * np.pi * df['Timestamp'].dt.dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df['Timestamp'].dt.dayofweek / 7)
    df["month_sin"] = np.sin(2 * np.pi * df['Timestamp'].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df['Timestamp'].dt.month / 12)
    df["weekend"] = (df['Timestamp'].dt.dayofweek >= 5).astype(int)
    
    # Wind Direction Cyclical Encoding
    if WIND_DIR_COL in df.columns:
        df["wind_sin"] = np.sin(2 * np.pi * df[WIND_DIR_COL] / 360)
        df["wind_cos"] = np.cos(2 * np.pi * df[WIND_DIR_COL] / 360)

    # Lags and Rolling Means
    exclude = ['Timestamp', TARGET_COL, WIND_DIR_COL, 'Date']
    weather_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    
    new_feats = {}
    for col in weather_cols:
        for lag in LAG_DAYS:
            new_feats[f"{col}_lag{lag}"] = df[col].shift(lag)
        for roll in ROLL_WINDOWS:
            new_feats[f"{col}_roll{roll}"] = df[col].shift(1).rolling(roll).mean()
            
    new_feats["admission_lag7"] = df[TARGET_COL].shift(7)
    df_final = pd.concat([df, pd.DataFrame(new_feats)], axis=1)
    return df_final.dropna().reset_index(drop=True)

# --- 2. Model Wrappers (Log Target & Keras) ---
class RidgeLogRegressor:
    def __init__(self):
        self.model = RidgeCV(alphas=np.logspace(-3, 2, 100))
    def fit(self, X, y):
        self.model.fit(X, np.log1p(y))
    def predict(self, X):
        return np.expm1(self.model.predict(X))

def build_keras_model(n_feat, params):
    model = keras.Sequential()
    model.add(layers.Input(shape=(n_feat,)))
    for _ in range(params.get('n_layers', 2)):
        model.add(layers.Dense(params.get('units', 64), activation='relu'))
        model.add(layers.Dropout(params.get('dropout', 0.2)))
    model.add(layers.Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=params.get('lr', 1e-3)), loss='mse')
    return model

# --- 3. Full Optuna Tuning Suite ---
def tune_models(X_tr, y_tr, X_va, y_va):
    # Ensure numpy inputs for all models to avoid the DataFrame .dtype error
    X_tr, y_tr = np.array(X_tr), np.array(y_tr).flatten()
    X_va, y_va = np.array(X_va), np.array(y_va).flatten()
    params_dict = {}

    # Define optimization objectives for each model type
    def optimize_model(m_type):
        def objective(trial):
            if m_type == 'rf':
                p = {'n_estimators': trial.suggest_int('n_estimators', 50, 200), 'max_depth': trial.suggest_int('max_depth', 5, 20)}
                m = RandomForestRegressor(**p, random_state=RANDOM_STATE)
            elif m_type == 'gbm':
                p = {'n_estimators': trial.suggest_int('n_estimators', 50, 200), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1)}
                m = GradientBoostingRegressor(**p, random_state=RANDOM_STATE)
            elif m_type == 'xgb':
                p = {'n_estimators': trial.suggest_int('n_estimators', 50, 200), 'max_depth': trial.suggest_int('max_depth', 3, 9), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1)}
                m = xgb.XGBRegressor(**p, random_state=RANDOM_STATE)
            elif m_type == 'svr':
                p = {'C': trial.suggest_float('C', 0.1, 10.0, log=True), 'epsilon': trial.suggest_float('epsilon', 0.01, 0.2)}
                m = SVR(**p)
            elif m_type == 'keras':
                p = {'units': trial.suggest_categorical('units', [32, 64, 128]), 'n_layers': trial.suggest_int('n_layers', 1, 3), 'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True)}
                m_keras = build_keras_model(X_tr.shape[1], p)
                m_keras.fit(X_tr, y_tr, epochs=50, verbose=0)
                return r2_score(y_va, m_keras.predict(X_va).flatten())

            m.fit(X_tr, y_tr)
            return r2_score(y_va, m.predict(X_va))
        
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=N_TRIALS)
        return study.best_params

    for m in ['rf', 'gbm', 'xgb', 'svr', 'keras']:
        print(f"  Tuning {m}...")
        params_dict[m] = optimize_model(m)
    return params_dict

# --- 4. Cross-Validation Loop ---
print("Initializing Data...")
df = engineer_features(pd.read_csv("/Users/suhaniagarwal/Downloads/all_features_data.csv"))
feat_cols = [c for c in df.columns if any(s in c for s in ["_lag", "_roll", "sin", "cos", "weekend"])]
X, y = df[feat_cols], df[TARGET_COL]

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
fold_results = []
best_params = {}



for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    print(f"\n--- FOLD {fold+1} ---")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Impute and Winsorize
    if USE_IMPUTATION:
        filler = X_train.median()
        X_train, X_test = X_train.fillna(filler), X_test.fillna(filler)
    
    y_train_capped = y_train.clip(upper=y_train.quantile(WINSORIZE_QUANTILE))
    scaler = StandardScaler()
    X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
    
    if fold == 0:
        split = int(len(X_train_s) * 0.8)
        best_params = tune_models(X_train_s[:split], y_train_capped[:split], X_train_s[split:], y_train_capped[split:])

    models = {
        "Dummy": DummyRegressor(strategy="mean"),
        "Ridge": RidgeCV(alphas=np.logspace(-3, 2, 100)),
        "RidgeLog": RidgeLogRegressor(),
        "ElasticNet": ElasticNetCV(cv=5),
        "RandomForest": RandomForestRegressor(**best_params['rf']),
        "GBM": GradientBoostingRegressor(**best_params['gbm']),
        "XGBoost": xgb.XGBRegressor(**best_params['xgb']),
        "SVR": SVR(**best_params['svr']),
        "Keras": None # Handled separately
    }

    scores = {}
    y_tr_arr, y_te_arr = np.array(y_train_capped).flatten(), np.array(y_test).flatten()

    for name, m in models.items():
        if name == "Keras":
            m_keras = build_keras_model(X_train_s.shape[1], best_params['keras'])
            m_keras.fit(X_train_s, y_tr_arr, epochs=100, verbose=0)
            preds = m_keras.predict(X_test_s).flatten()
        else:
            m.fit(X_train_s, y_tr_arr)
            preds = m.predict(X_test_s)
        
        scores[name] = r2_score(y_te_arr, preds)
        print(f"  {name}: {scores[name]:.4f}")
    
    fold_results.append(scores)

print("\n" + "="*30 + "\nFinal Average R2 Scores:\n" + "="*30)
print(pd.DataFrame(fold_results).mean())

Initializing Data...

--- FOLD 1 ---
  Tuning rf...
  Tuning gbm...
  Tuning xgb...
  Tuning svr...
  Tuning keras...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━

In [47]:
import os
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


USE_IMPUTATION = True
WINSORIZE_QUANTILE = 0.99 
LAG_DAYS = [1, 3, 7]
ROLL_WINDOWS = [3, 7]
TARGET_COL = "Number of Admissions"
WIND_DIR_COL = "wind_direction_10m_dominant (°)"
N_SPLITS = 5
N_TRIALS = 25
RANDOM_STATE = 42

# --- 1. FEATURE ENGINEERING ---
def engineer_features(df):
    df = df.copy()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.sort_values('Timestamp').reset_index(drop=True)
    
    # Cyclical Time Features
    df["dow_sin"] = np.sin(2 * np.pi * df['Timestamp'].dt.dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df['Timestamp'].dt.dayofweek / 7)
    df["month_sin"] = np.sin(2 * np.pi * df['Timestamp'].dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * df['Timestamp'].dt.month / 12)
    df["weekend"] = (df['Timestamp'].dt.dayofweek >= 5).astype(int)
    
    # Wind Direction Encoding (prevents 0/360 discontinuity)
    if WIND_DIR_COL in df.columns:
        df["wind_sin"] = np.sin(2 * np.pi * df[WIND_DIR_COL] / 360)
        df["wind_cos"] = np.cos(2 * np.pi * df[WIND_DIR_COL] / 360)

    # Historic Lags and Rolling Means
    exclude = ['Timestamp', TARGET_COL, WIND_DIR_COL, 'Date']
    weather_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    
    new_feats = {}
    for col in weather_cols:
        for lag in LAG_DAYS:
            new_feats[f"{col}_lag{lag}"] = df[col].shift(lag)
        for roll in ROLL_WINDOWS:
            new_feats[f"{col}_roll{roll}"] = df[col].shift(1).rolling(roll).mean()
            
    new_feats["admission_lag7"] = df[TARGET_COL].shift(7)
    return pd.concat([df, pd.DataFrame(new_feats)], axis=1).dropna().reset_index(drop=True)

# --- 2. WRAPPERS & KERAS SETUP ---
class RidgeLogRegressor:
    def __init__(self):
        self.model = RidgeCV(alphas=np.logspace(-3, 2, 100), cv=5)
    def fit(self, X, y):
        self.model.fit(X, np.log1p(y))
    def predict(self, X):
        return np.expm1(self.model.predict(X))

def build_keras_model(n_feat, params):
    model = keras.Sequential([
        layers.Input(shape=(n_feat,)),
        *[layers.Dense(params['units'], activation='relu') for _ in range(params['n_layers'])],
        layers.Dropout(params['dropout']),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=params['lr']), loss='mse')
    return model


def get_optuna_params(model_type, X_tr, y_tr, X_va, y_va):
    y_tr, y_va = np.array(y_tr).flatten(), np.array(y_va).flatten()
    
    def objective(trial):
        if model_type == 'rf':
            p = {'n_estimators': trial.suggest_int("n_estimators", 50, 200),
                 'max_depth': trial.suggest_int("max_depth", 5, 20),
                 'min_samples_leaf': trial.suggest_int("min_samples_leaf", 2, 20)}
            model = RandomForestRegressor(**p, random_state=RANDOM_STATE, n_jobs=-1)
        elif model_type == 'gbm':
            p = {'n_estimators': trial.suggest_int("n_estimators", 50, 300),
                 'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                 'max_depth': trial.suggest_int("max_depth", 2, 8)}
            model = GradientBoostingRegressor(**p, random_state=RANDOM_STATE)
        elif model_type == 'histgb':
            p = {'max_iter': trial.suggest_int("max_iter", 50, 250),
                 'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.2, log=True)}
            model = HistGradientBoostingRegressor(**p, random_state=RANDOM_STATE)
        elif model_type == 'et':
            p = {'n_estimators': trial.suggest_int("n_estimators", 50, 250),
                 'max_depth': trial.suggest_int("max_depth", 5, 20)}
            model = ExtraTreesRegressor(**p, random_state=RANDOM_STATE, n_jobs=-1)
        elif model_type == 'xgb':
            p = {'n_estimators': trial.suggest_int("n_estimators", 50, 300),
                 'max_depth': trial.suggest_int("max_depth", 3, 10),
                 'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.2, log=True)}
            model = xgb.XGBRegressor(**p, random_state=RANDOM_STATE)
        elif model_type == 'svr':
            p = {'C': trial.suggest_float("C", 0.1, 100.0, log=True),
                 'epsilon': trial.suggest_float("epsilon", 0.01, 1.0)}
            model = SVR(**p)
        elif model_type == 'mlp':
            n_layers = trial.suggest_int("n_layers", 1, 3)
            units = trial.suggest_int("units", 16, 128)
            model = MLPRegressor(hidden_layer_sizes=(units,)*n_layers, max_iter=1000, random_state=RANDOM_STATE)
        elif model_type == 'keras':
            p = {'units': trial.suggest_categorical("units", [16, 32, 64, 128]),
                 'n_layers': trial.suggest_int("n_layers", 1, 3),
                 'dropout': trial.suggest_float("dropout", 0.2, 0.5),
                 'lr': trial.suggest_float("lr", 3e-4, 3e-3, log=True)}
            model = build_keras_model(X_tr.shape[1], p)
            model.fit(X_tr, y_tr, epochs=50, verbose=0)
            return r2_score(y_va, model.predict(X_va).flatten())

        model.fit(X_tr, y_tr)
        return r2_score(y_va, model.predict(X_va).flatten())

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    return study.best_params

# --- 4. MAIN CROSS-VALIDATION LOOP ---
df = engineer_features(pd.read_csv("/Users/suhaniagarwal/Downloads/all_features_data.csv"))
feat_cols = [c for c in df.columns if any(s in c for s in ["_lag", "_roll", "sin", "cos", "weekend"])]
X, y = df[feat_cols], df[TARGET_COL]

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
fold_results = []
best_params = {}



for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    if USE_IMPUTATION:
        filler = X_train.median()
        X_train, X_test = X_train.fillna(filler), X_test.fillna(filler)
    
    y_train_capped = y_train.clip(upper=y_train.quantile(WINSORIZE_QUANTILE))
    scaler = StandardScaler()
    X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)
    
    # Tune once on Fold 1
    if fold == 0:
        print("Tuning models on Fold 1...")
        split = int(len(X_train_s) * 0.8)
        for m in ['rf', 'gbm', 'histgb', 'et', 'xgb', 'svr', 'mlp', 'keras']:
            best_params[m] = get_optuna_params(m, X_train_s[:split], y_train_capped[:split], 
                                                X_train_s[split:], y_train_capped[split:])

    # FINAL MODEL DEFINITIONS
    y_train_arr, y_test_arr = np.array(y_train_capped).flatten(), np.array(y_test).flatten()
    models = {
        "Dummy (mean)": DummyRegressor(strategy="mean"),
        "Ridge": RidgeCV(alphas=np.logspace(-3, 2, 100), cv=5),
        "RidgeLog": RidgeLogRegressor(),
        "ElasticNet": ElasticNetCV(cv=5, l1_ratio=[0.1, 0.5, 0.9, 1.0]),
        "RandomForest": RandomForestRegressor(**best_params['rf']),
        "GBM": GradientBoostingRegressor(**best_params['gbm']),
        "HistGB": HistGradientBoostingRegressor(**best_params['histgb']),
        "ExtraTrees": ExtraTreesRegressor(**best_params['et']),
        "XGBoost": xgb.XGBRegressor(**best_params['xgb']),
        "SVR": SVR(**best_params['svr']),
        "MLP": MLPRegressor(hidden_layer_sizes=(best_params['mlp']['units'],)*best_params['mlp'].get('n_layers', 1))
    }
    
    scores = {}
    for name, model in models.items():
        model.fit(X_train_s, y_train_arr)
        scores[name] = r2_score(y_test_arr, model.predict(X_test_s).flatten())
        
    # Keras fit separately
    m_keras = build_keras_model(X_train_s.shape[1], best_params['keras'])
    m_keras.fit(X_train_s, y_train_arr, epochs=100, verbose=0)
    scores["Keras"] = r2_score(y_test_arr, m_keras.predict(X_test_s).flatten())
    
    fold_results.append(scores)
    print(f"Fold {fold+1} complete.")

# Final Average Results
print("\nFinal Average R2 Scores (5-Fold CV):")
print(pd.DataFrame(fold_results).mean())

Tuning models on Fold 1...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
4/4 ━━━━━━━━━━━━━━━━━━━